# 얼굴 표정 7종 분류 · ViT-LoRA

얼굴이 중심인 사진 한 장을 입력해 FER2013의 표정 클래스와 점수를 출력합니다. 같은 ViT에서 7종 분류기를 초기화한 직후와 Q·V LoRA 및 분류기를 학습한 뒤의 성능을 비교합니다.

이 노트북은 Google Colab의 CPU 런타임에서 실행했습니다. Colab에서 다시 실행하려면 2절 설치 셀을 실행한 뒤 위에서부터 순서대로 실행합니다. 패키지 설치 뒤 import 오류가 나면 세션을 재시작하고 과제 설정 셀부터 다시 실행합니다.

## 1. 과제 설정

**실험 질문** · 얼굴 표정 7종을 분류할 때, 사전학습 ViT에 LoRA와 새 분류기를 학습하면 같은 테스트셋에서 Accuracy, Macro F1, 클래스별 오분류가 어떻게 달라지는가?

In [ ]:
TASK = "Vision"
MODEL_ID = "WinKawaks/vit-small-patch16-224"
MODEL_REVISION = "0c08c1829024fa177bd69b6dd09b05ad980854e1"
DATASET_ID = "Aaryan333/fer2013_train_publicTest_privateTest"
DATASET_REVISION = "da8d1643649c86bfc2cffba5b744b51fdf329212"
SEED = 42
OUTPUT_DIR = "./assignment_runs/face_expression"

# 분할별 층화 표본의 최대 크기입니다. None은 해당 분할 전체를 사용합니다.
MAX_EXAMPLES = {"train": 700, "validation": 210, "test": 350}
TRAIN_EPOCHS = 2

ViT-Small은 예제와 같은 ImageNet 사전학습 모델입니다. Colab에서 다루기 쉬운 규모이고, Self-Attention의 Q·V 투영에 LoRA를 적용할 수 있습니다. 모델 카드에 Apache-2.0 라이선스가 표시되어 있습니다. [모델 카드](https://huggingface.co/WinKawaks/vit-small-patch16-224)

FER2013의 48×48 회색조 얼굴 이미지와 분노·혐오·두려움·기쁨·슬픔·놀람·중립 레이블을 사용합니다. 공개 Hub 미러는 원본의 Train/PublicTest/PrivateTest에 해당하는 파일 세 개를 제공합니다. 미러 카드에는 개별 라이선스 정보가 기재되어 있지 않습니다. [Hub 데이터](https://huggingface.co/datasets/Aaryan333/fer2013_train_publicTest_privateTest) · [원 논문](https://arxiv.org/abs/1307.0414)

## 2. 실행 환경

주요 라이브러리 버전을 고정하고 Colab의 PyTorch와 torchvision을 사용합니다. 이미지 전처리에는 이번 Colab 런타임의 Pillow 11.3.0을 사용했습니다. ViT-LoRA 학습은 양자화를 사용하지 않습니다. Colab의 기본 `torchao`가 이번 PyTorch 환경에서 의존성 충돌을 일으켜 설치 셀에서 제거했습니다. 아래 출력에서 사용한 버전과 실행 환경을 확인할 수 있습니다.

In [13]:
%pip install transformers==5.23.0 datasets==5.0.0 peft==0.21.3 accelerate==1.16.0 huggingface-hub==1.32.0 fsspec==2025.12.0 pyarrow==23.0.1 Pillow==11.3.0 pandas==2.2.3
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [14]:
import importlib.metadata
import json
import os
from hashlib import sha256
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict, load_dataset
from IPython.display import display
from peft import LoraConfig, PeftModel, get_peft_model
from PIL import Image, ImageOps
from transformers import (
    AutoConfig,
    AutoImageProcessor,
    AutoModelForImageClassification,
    Trainer,
    TrainingArguments,
    default_data_collator,
    pipeline,
    set_seed,
)

set_seed(SEED)
print("실행 환경:", "Google Colab" if "COLAB_RELEASE_TAG" in os.environ else "기타 환경")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
for package in ("torch", "torchvision", "transformers", "datasets", "peft", "accelerate",
                "huggingface-hub", "fsspec", "pyarrow", "pillow", "pandas"):
    print(package, importlib.metadata.version(package))


def expression_image(image):
    return ImageOps.exif_transpose(image).convert("L").convert("RGB")


def make_image_pipeline(current_model, processor):
    current_model.eval()
    # 예제와 같이 LoRA가 삽입된 ViT 본체를 Pipeline에 전달합니다.
    base = current_model.get_base_model() if isinstance(current_model, PeftModel) else current_model
    return pipeline(
        "image-classification", model=base, image_processor=processor,
        device=0 if torch.cuda.is_available() else -1,
    )


def reload_classifier(save_path):
    saved = json.loads((save_path / "experiment.json").read_text(encoding="utf-8"))
    config = AutoConfig.from_pretrained(save_path)
    base = AutoModelForImageClassification.from_pretrained(
        saved["base_model"], revision=saved["model_revision"], config=config,
        use_safetensors=True, ignore_mismatched_sizes=True,
    )
    restored = PeftModel.from_pretrained(base, save_path)
    restored_processor = AutoImageProcessor.from_pretrained(save_path, use_fast=False)
    return restored, restored_processor, saved["labels_ko"]

실행 환경: Google Colab
GPU: CPU
torch 2.14.1+cpu
torchvision 0.29.1+cpu
transformers 5.23.0
datasets 5.0.0
peft 0.21.3
accelerate 1.16.0
huggingface-hub 1.32.0
fsspec 2025.12.0
pyarrow 23.0.1
pillow 11.3.0
pandas 2.2.3


## 3. 데이터 로딩과 분할

Train → 학습, PublicTest → 검증, PrivateTest → 최종 테스트로 사용합니다. 샘플을 다른 원본 분할로 옮기지 않습니다.
같은 크기와 회색조 픽셀을 갖는 이미지는 Test → Validation → Train 순서로 한 번만 남깁니다. 그 뒤 각 분할에서 클래스 비율을 유지하며 고정 시드로 표본을 뽑습니다.
이 검사의 범위는 동일 픽셀 중복이며, 같은 인물의 다른 사진과 비슷한 크롭은 남을 수 있습니다.

In [15]:
from datasets import Dataset, DatasetDict

def prepare_splits(data, seed=SEED):
    splits = DatasetDict({"train": data}) if isinstance(data, Dataset) else DatasetDict(data)
    if "train" not in splits:
        raise ValueError("학습용 분할의 이름을 train으로 지정하세요.")
    if "test" not in splits:
        split = splits["train"].train_test_split(test_size=0.1, seed=seed)
        splits["train"], splits["test"] = split["train"], split["test"]
        validation_ratio = 1 / 9
    else:
        validation_ratio = 0.1
    if "validation" not in splits:
        split = splits["train"].train_test_split(test_size=validation_ratio, seed=seed)
        splits["train"], splits["validation"] = split["train"], split["test"]
    return DatasetDict({name: splits[name] for name in ("train", "validation", "test")})


def load_data():
    source = load_dataset(DATASET_ID, revision=DATASET_REVISION)
    return DatasetDict({
        "train": source["train"],
        "validation": source["publicTest"],
        "test": source["privateTest"],
    })


def fingerprint_images(batch):
    fingerprints = []
    for image in batch["image"]:
        gray = image.convert("L")
        digest = sha256(f"{gray.width}x{gray.height}:".encode() + gray.tobytes())
        fingerprints.append(digest.hexdigest())
    return {"image_hash": fingerprints}


def deduplicate_splits(splits):
    seen = set()
    cleaned = {}
    for split_name in ("test", "validation", "train"):
        data = splits[split_name]
        hashes = data.select_columns("image").map(
            fingerprint_images, batched=True, batch_size=128,
            remove_columns=["image"], desc=f"{split_name} 중복 확인",
        )["image_hash"]
        keep = []
        for index, image_hash in enumerate(hashes):
            if image_hash not in seen:
                keep.append(index)
                seen.add(image_hash)
        cleaned[split_name] = data.select(keep)
    return DatasetDict({name: cleaned[name] for name in splits})


source_splits = prepare_splits(load_data())
source_labels = source_splits["train"].features["label"].names
assert all(data.features["label"].names == source_labels for data in source_splits.values())
labels = [label.strip().lower() for label in source_labels]
label_ko = {
    "angry": "분노", "disgust": "혐오", "fear": "두려움", "happy": "기쁨",
    "sad": "슬픔", "surprise": "놀람", "neutral": "중립",
}
assert len(labels) == 7 and set(labels) == set(label_ko), labels
id2label = dict(enumerate(labels))
label2id = {label: index for index, label in id2label.items()}
id2ko = {index: label_ko[label] for index, label in id2label.items()}

unique_splits = deduplicate_splits(source_splits)
raw = DatasetDict()
for name, data in unique_splits.items():
    limit = MAX_EXAMPLES[name]
    if limit is not None and limit < len(data):
        data = data.train_test_split(
            train_size=limit, stratify_by_column="label", seed=SEED
        )["train"]
    assert set(data["label"]) == set(id2label), f"{name}에 빠진 클래스가 있습니다."
    raw[name] = data

split_summary = pd.DataFrame({
    "원본": {name: len(data) for name, data in source_splits.items()},
    "중복 제거 후": {name: len(data) for name, data in unique_splits.items()},
    "사용": {name: len(data) for name, data in raw.items()},
})
display(split_summary)
display(pd.DataFrame({
    name: pd.Series(list(data["label"])).value_counts().reindex(id2label, fill_value=0)
    for name, data in raw.items()
}).rename(index=id2ko))
print("첫 학습 샘플의 표정:", id2ko[raw["train"][0]["label"]])
display(raw["train"][0]["image"].resize((144, 144)))

,원본,중복 제거 후,사용
train,28709,26942,700
validation,3589,3520,210
test,3589,3572,350


,train,validation,test
분노,98,27,48
혐오,9,3,6
두려움,99,29,51
기쁨,183,53,86
슬픔,122,39,58
놀람,64,23,40
중립,125,36,61


첫 학습 샘플의 표정: 기쁨


## 4. 모델과 전처리

`map()`으로 학습·검증 사진을 회색조 RGB로 바꾸고 `labels`를 만듭니다. 데이터셋이 배치를 읽을 때 이미지 프로세서가 224×224 크기 조정과 정규화를 수행합니다. 테스트와 사용자가 넣는 사진에도 같은 이미지 프로세서를 적용합니다.

In [16]:
def load_model_and_processor():
    processor = AutoImageProcessor.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, use_fast=False,
    )
    model = AutoModelForImageClassification.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, use_safetensors=True,
        num_labels=len(labels), id2label=id2label, label2id=label2id,
        ignore_mismatched_sizes=True,
    )
    return model, processor


model, processor = load_model_and_processor()


def preprocess(batch):
    return {
        "image": [expression_image(image) for image in batch["image"]],
        "labels": batch["label"],
    }


def make_pixel_batch(batch):
    pixels = processor(images=batch["image"], return_tensors="pt")["pixel_values"]
    return {"pixel_values": pixels, "labels": batch["labels"]}


prepared = DatasetDict({name: raw[name] for name in ("train", "validation")}).map(
    preprocess, batched=True, batch_size=16, writer_batch_size=16,
    remove_columns=["label"], desc="ViT 입력 전처리",
)
for data in prepared.values():
    data.set_transform(make_pixel_batch)
pixels = prepared["train"][0]["pixel_values"]
assert tuple(pixels.shape) == (3, 224, 224), pixels.shape
print("모델 입력:", pixels.shape, pixels.dtype)
print("레이블:", id2label)
print("정규화 평균·표준편차:", processor.image_mean, processor.image_std)

[transformers] You passed `num_labels=7` which is incompatible to the `id2label` map of length `1000`.


[transformers] ViTForImageClassification LOAD REPORT from: WinKawaks/vit-small-patch16-224
Key               | Status   |
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 384]) vs model:torch.Size([7, 384])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([7])

Notes:
- MISMATCH: ckpt weights were loaded, but they did not match the original empty weight shapes.


모델 입력: torch.Size([3, 224, 224]) torch.float32
레이블: {0: 'angry', 1: 'disgust', 2: 'fear', 3: 'happy', 4: 'sad', 5: 'surprise', 6: 'neutral'}
정규화 평균·표준편차: (0.5, 0.5, 0.5) (0.5, 0.5, 0.5)


ViT-Small의 기존 ImageNet 분류기를 7개 출력으로 새로 초기화합니다. 불러올 때 분류기 가중치 크기가 달라졌다는 메시지는 이 변경에 해당합니다.
`224×224` 입력은 `16×16` 이미지 패치 196개로 바뀝니다. ViT는 위치 정보를 더하고, Self-Attention으로 패치 사이의 정보를 결합한 뒤 `[CLS]` 표현을 분류에 사용합니다. [ViT 논문](https://arxiv.org/abs/2010.11929)

## 5. 학습 전 평가

사전학습 본체와 아직 학습하지 않은 7종 분류기의 초기 상태를 평가합니다. `pipeline`에서 얻은 영문 예측을 정답과 같은 정수 ID로 변환합니다. 고정 테스트셋 전체에서 Accuracy와 Macro F1을 계산합니다.

In [17]:
def get_targets(test_data):
    return list(test_data["label"])


def predict_labels(current_model, processor, test_data):
    classifier = make_image_pipeline(current_model, processor)
    images = [expression_image(image) for image in test_data["image"]]
    outputs = classifier(images, top_k=1, batch_size=16)
    return [label2id[output[0]["label"]] for output in outputs]


def expression_confusion(targets, predictions):
    matrix = np.zeros((len(labels), len(labels)), dtype=np.int64)
    np.add.at(matrix, (np.asarray(targets, dtype=int), np.asarray(predictions, dtype=int)), 1)
    return matrix


def classification_metrics(targets, predictions):
    score = accuracy(targets, predictions)
    matrix = expression_confusion(targets, predictions)
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    f1 = np.divide(2 * matrix.diagonal(), denominator,
                   out=np.zeros(len(labels), dtype=float), where=denominator != 0)
    return {"accuracy": score, "macro_f1": float(f1.mean())}


def accuracy(targets, predictions):
    if len(targets) == 0 or len(targets) != len(predictions):
        raise ValueError("정답과 예측은 길이가 같고 비어 있지 않아야 합니다.")
    return sum(a == b for a, b in zip(targets, predictions)) / len(targets)

def evaluate_model(current_model, processor, test_data):
    targets = get_targets(test_data)
    predictions = predict_labels(current_model, processor, test_data)
    if len(targets) != len(test_data):
        raise ValueError("Test의 모든 샘플에 대해 정답과 예측을 반환하세요.")
    metrics = classification_metrics(targets, predictions)
    table = pd.DataFrame({"sample_id": range(len(targets)), "target": targets,
                          "prediction": predictions})
    return metrics, table

before_metrics, before = evaluate_model(model, processor, raw["test"])
print(before_metrics)
fixed_test_samples = before.drop_duplicates("target").sort_values("target")
fixed_test_ids = fixed_test_samples["sample_id"].tolist()
for row in fixed_test_samples.itertuples(index=False):
    print(f"Test #{row.sample_id} · 정답 {id2ko[row.target]} · 학습 전 {id2ko[row.prediction]}")
    display(raw["test"][row.sample_id]["image"].resize((144, 144)))


{'accuracy': 0.10571428571428572, 'macro_f1': 0.07938635917181261}
Test #15 · 정답 분노 · 학습 전 놀람
Test #139 · 정답 혐오 · 학습 전 분노
Test #40 · 정답 두려움 · 학습 전 혐오
Test #0 · 정답 기쁨 · 학습 전 놀람
Test #11 · 정답 슬픔 · 학습 전 놀람
Test #2 · 정답 놀람 · 학습 전 놀람
Test #1 · 정답 중립 · 학습 전 혐오


## 6. LoRA 적용

Query와 Key의 유사도로 Attention 가중치를 만들고, 이 가중치로 Value 정보를 합칩니다. 이번 실험에서는 Q·V의 원래 가중치를 고정하고 저랭크 업데이트 `ΔW = (α/r)BA`를 학습합니다. 새 분류기도 학습 및 저장 대상에 포함합니다. [LoRA 논문](https://arxiv.org/abs/2106.09685) · [PEFT 이미지 분류 가이드](https://huggingface.co/docs/peft/main/task_guides/image_classification_lora)

In [18]:
def make_lora_config(model):
    linear_names = {
        name.rsplit(".", 1)[-1]
        for name, module in model.named_modules() if isinstance(module, torch.nn.Linear)
    }
    if {"q_proj", "v_proj"} <= linear_names:
        targets = ["q_proj", "v_proj"]
    elif {"query", "value"} <= linear_names:
        targets = ["query", "value"]
    else:
        raise ValueError("model.named_modules()에서 ViT의 Query/Value 이름을 확인하세요.")
    print("LoRA 대상:", targets)
    return LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.1, bias="none",
        target_modules=targets, modules_to_save=["classifier"],
    )


lora_config = make_lora_config(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
trainable_names = [name for name, param in model.named_parameters() if param.requires_grad]
assert any("lora_" in name for name in trainable_names)
assert any("classifier" in name for name in trainable_names)
print("학습 대상 예:", trainable_names[:2] + trainable_names[-2:])

LoRA 대상: ['q_proj', 'v_proj']
trainable params: 150,151 || all params: 21,818,510 || trainable%: 0.6882
학습 대상 예: ['base_model.model.vit.layers.0.attention.q_proj.lora_A.default.weight', 'base_model.model.vit.layers.0.attention.q_proj.lora_B.default.weight', 'base_model.model.classifier.modules_to_save.default.weight', 'base_model.model.classifier.modules_to_save.default.bias']


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


랭크는 8, 스케일은 `alpha=16`, 어댑터 드롭아웃은 0.1입니다. Q·V 어댑터와 새 분류기를 함께 학습하므로 전후 차이에 두 요소의 효과가 함께 포함됩니다.

## 7. 학습

초기 학습률 `5e-4`, 설정 셀의 epoch 수, 배치 16을 사용합니다. 매 epoch의 Validation Macro F1로 체크포인트를 선택합니다. Test는 Trainer에 전달하지 않습니다.

In [19]:
def make_trainer(model, processor, train_data, validation_data):
    args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        learning_rate=5e-4,
        num_train_epochs=TRAIN_EPOCHS,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        label_names=["labels"],
        remove_unused_columns=False,
        report_to="none",
        seed=SEED,
        logging_steps=20,
    )

    def compute_metrics(result):
        return classification_metrics(
            result.label_ids.tolist(), result.predictions.argmax(-1).tolist()
        )

    return Trainer(
        model=model, args=args, train_dataset=train_data, eval_dataset=validation_data,
        processing_class=processor, data_collator=default_data_collator,
        compute_metrics=compute_metrics,
    )


trainer = make_trainer(model, processor, prepared["train"], prepared["validation"])
trainer.train()
print("선택된 체크포인트:", trainer.state.best_model_checkpoint)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[88/88 10:43, Epoch 2/2]

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.632944,1.585490,0.366667,0.252133
2,1.336131,1.489021,0.400000,0.288528


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


선택된 체크포인트: ./assignment_runs/face_expression/checkpoint-88


## 8. 학습 후 평가

같은 테스트 이미지에 동일한 추론 함수를 적용합니다. Accuracy, 7개 클래스의 F1을 동일한 비중으로 평균한 Macro F1, 클래스별 정답률을 기록합니다.
비교 예시는 정답 클래스별 첫 샘플로 고정해 일곱 장을 출력합니다.

In [20]:
after_metrics, after = evaluate_model(model, processor, raw["test"])
assert len(before) >= 5 and len(after) >= 5, "비교 샘플을 최소 5개 기록하세요."
assert before["sample_id"].is_unique and after["sample_id"].is_unique
assert before[["sample_id", "target"]].reset_index(drop=True).equals(
    after[["sample_id", "target"]].reset_index(drop=True)
), "학습 전후 동일한 샘플·순서·정답을 사용하세요."
comparison = before.rename(columns={"prediction": "before"}).merge(
    after[["sample_id", "prediction"]].rename(columns={"prediction": "after"}),
    on="sample_id", validate="one_to_one"
)
metrics_table = pd.DataFrame({"학습 전": before_metrics, "학습 후": after_metrics})
metrics_table["변화 (%p)"] = (metrics_table["학습 후"] - metrics_table["학습 전"]) * 100
display(metrics_table.round(4))

samples = comparison.set_index("sample_id").loc[fixed_test_ids].reset_index()
sample_table = samples.copy()
for column in ("target", "before", "after"):
    sample_table[column] = sample_table[column].map(id2ko)
display(sample_table.rename(columns={"sample_id": "Test 위치", "target": "정답",
                                     "before": "학습 전", "after": "학습 후"}))
for row in samples.itertuples(index=False):
    print(f"Test #{row.sample_id} · 정답 {id2ko[row.target]} · 전 {id2ko[row.before]} · 후 {id2ko[row.after]}")
    display(raw["test"][row.sample_id]["image"].resize((144, 144)))

confusion = expression_confusion(comparison["target"], comparison["after"])
confusion_table = pd.DataFrame(confusion, index=id2ko.values(), columns=id2ko.values())
display(confusion_table.rename_axis(index="정답", columns="학습 후 예측"))
recall = pd.Series(confusion.diagonal() / confusion.sum(axis=1), index=id2ko.values())
display(pd.DataFrame({"Test 개수": confusion.sum(axis=1), "클래스별 정답률": recall.round(4)}))

before_correct = comparison["before"] == comparison["target"]
after_correct = comparison["after"] == comparison["target"]
print(
    f"Test {len(comparison)}개에서 Accuracy {before_metrics['accuracy']:.2%} → "
    f"{after_metrics['accuracy']:.2%} ({metrics_table.at['accuracy', '변화 (%p)']:+.2f}%p). "
    f"오답→정답 {int((~before_correct & after_correct).sum())}개, "
    f"정답→오답 {int((before_correct & ~after_correct).sum())}개."
)
failures = comparison.loc[~after_correct]
if not failures.empty:
    target, prediction = failures.groupby(["target", "after"]).size().idxmax()
    examples = failures.loc[(failures["target"] == target) & (failures["after"] == prediction)]
    print(f"가장 잦은 혼동: {id2ko[target]} → {id2ko[prediction]} ({len(examples)}개)")
    for index in examples["sample_id"].head(2):
        print(f"Test #{index}")
        display(raw["test"][int(index)]["image"].resize((144, 144)))
else:
    print("이번 테스트셋에서 학습 후 오분류 0개")

,학습 전,학습 후,변화 (%p)
accuracy,0.1057,0.4629,35.7143
macro_f1,0.0794,0.3558,27.6451


,Test 위치,정답,학습 전,학습 후
0,15,분노,놀람,기쁨
1,139,혐오,분노,분노
2,40,두려움,혐오,기쁨
3,0,기쁨,놀람,기쁨
4,11,슬픔,놀람,기쁨
5,2,놀람,놀람,놀람
6,1,중립,혐오,중립


Test #15 · 정답 분노 · 전 놀람 · 후 기쁨
Test #139 · 정답 혐오 · 전 분노 · 후 분노
Test #40 · 정답 두려움 · 전 혐오 · 후 기쁨
Test #0 · 정답 기쁨 · 전 놀람 · 후 기쁨
Test #11 · 정답 슬픔 · 전 놀람 · 후 기쁨
Test #2 · 정답 놀람 · 전 놀람 · 후 놀람
Test #1 · 정답 중립 · 전 혐오 · 후 중립


학습 후 예측,분노,혐오,두려움,기쁨,슬픔,놀람,중립
정답,,,,,,,
분노,11,0,4,12,9,4,8
혐오,2,0,0,2,0,1,1
두려움,1,0,12,12,8,6,12
기쁨,1,0,4,70,4,3,4
슬픔,12,0,8,3,18,1,16
놀람,1,0,10,8,1,15,5
중립,8,0,4,7,3,3,36


,Test 개수,클래스별 정답률
분노,48,0.2292
혐오,6,0.0000
두려움,51,0.2353
기쁨,86,0.8140
슬픔,58,0.3103
놀람,40,0.3750
중립,61,0.5902


Test 350개에서 Accuracy 10.57% → 46.29% (+35.71%p). 오답→정답 144개, 정답→오답 19개.
가장 잦은 혼동: 슬픔 → 중립 (16개)
Test #22
Test #37


## 9. 저장과 재로딩

학습한 LoRA와 분류기, 프로세서, 레이블 매핑과 리비전을 저장합니다. 어댑터를 다시 불러와 같은 이미지에서 예측이 재현되는지 확인합니다.

In [21]:
from pathlib import Path

adapter_dir = Path(OUTPUT_DIR) / "adapter"
model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)
model.config.save_pretrained(adapter_dir)
print("Base model:", MODEL_ID)
print("Adapter·Processor·모델 설정 저장 위치:", adapter_dir.resolve())


metadata = {
    "base_model": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "labels_ko": label_ko,
    "seed": SEED,
    "epochs": TRAIN_EPOCHS,
    "examples": {name: len(data) for name, data in raw.items()},
    "before": before_metrics,
    "after": after_metrics,
}
(adapter_dir / "experiment.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("모델 리비전:", MODEL_REVISION, "· 데이터 리비전:", DATASET_REVISION)
print("저장 파일:", sorted(path.name for path in adapter_dir.iterdir()))


restored_model, restored_processor, restored_labels_ko = reload_classifier(adapter_dir)
selected_ids = samples["sample_id"].astype(int).tolist()
restored_predictions = predict_labels(
    restored_model, restored_processor, raw["test"].select(selected_ids)
)
assert restored_predictions == samples["after"].tolist(), "저장 전후 예측이 다릅니다."
print(f"재로딩 확인: 고정 테스트 예시 {len(selected_ids)}개의 예측 일치")

Base model: WinKawaks/vit-small-patch16-224
Adapter·Processor·모델 설정 저장 위치: /content/assignment_runs/face_expression/adapter
모델 리비전: 0c08c1829024fa177bd69b6dd09b05ad980854e1 · 데이터 리비전: da8d1643649c86bfc2cffba5b744b51fdf329212
저장 파일: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'config.json', 'experiment.json', 'preprocessor_config.json']


[transformers] ViTForImageClassification LOAD REPORT from: WinKawaks/vit-small-patch16-224
Key               | Status   |
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 384]) vs model:torch.Size([7, 384])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([7])

Notes:
- MISMATCH: ckpt weights were loaded, but they did not match the original empty weight shapes.

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


재로딩 확인: 고정 테스트 예시 7개의 예측 일치


### 결과 해석과 제출

Google Colab의 CPU 런타임에서 학습 700장, 검증 210장으로 2 epoch을 실행했습니다. 동일한 테스트 350장에서 Accuracy는 **10.57% → 46.29%**(+35.71%p), Macro F1은 **0.0794 → 0.3558**였습니다. 오답이 정답으로 바뀐 사진은 144장, 정답이 오답으로 바뀐 사진은 19장입니다. 두 변화를 합산하면 정답 수는 +125장 변했습니다.

오분류 건수 기준으로 가장 자주 나타난 표정 쌍은 **슬픔 → 중립 (16개)**였습니다. 이 집계는 표정별 테스트 개수의 영향을 받으므로 클래스별 정답률 표를 함께 해석했습니다. 학습 후 정답률은 기쁨에서 81.4%(86장), 혐오에서 0.0%(6장)였습니다. 표본이 적은 클래스의 정답률은 한두 장의 예측에 크게 달라질 수 있습니다.

Accuracy의 변화는 새 분류기와 Q·V LoRA를 함께 학습한 결과입니다. Macro F1과 클래스별 정답률을 함께 보면 샘플이 적은 표정에서도 성능이 개선됐는지 확인할 수 있습니다. 8절은 실제 실행값으로 전후 정답 여부의 변화와 자주 혼동된 표정 쌍을 출력합니다.

FER2013은 작은 회색조 얼굴 사진과 사람이 붙인 표정 레이블을 사용합니다. 얼굴 각도, 조명, 가림, 사진 해상도에 따라 결과가 달라질 수 있습니다. 모델 출력은 사진에서 관찰되는 표정 범주의 점수이며 사진 속 사람의 실제 감정에는 표정 외에도 상황과 개인차가 영향을 줍니다.

과제에는 코드, 실제 실행 출력, 설명과 분석이 함께 들어 있는 이 `.ipynb` 파일 하나를 제출합니다.

Colab 저장 충돌 후 실행 로그와 표를 복구했습니다. 표는 정적 HTML로 저장했고 설치 세부 로그와 이미지 미리보기의 출력은 정리했습니다. 셀을 다시 실행하면 이미지도 함께 표시됩니다.

## 10. 내 사진으로 표정 확인 · 선택

얼굴이 화면 중심에 있는 사진을 사용합니다. Colab에서 `UPLOAD_FROM_COLAB = True`로 바꾸면 업로드 창이 열립니다. 이미 올린 파일은 `IMAGE_PATH`에 경로를 적습니다. 둘 다 기본값이면 데이터셋 예시 한 장으로 동작을 확인합니다.

이미 저장한 어댑터로 사진만 분석하려면 설정 및 환경 import 셀을 실행한 후 아래 셀에서 `IMAGE_PATH`를 지정합니다. `OUTPUT_DIR/adapter`에 저장된 파일을 읽어 사진을 분류합니다.

In [22]:
IMAGE_PATH = ""  # 예: /content/face.jpg
UPLOAD_FROM_COLAB = False

if "restored_model" not in globals():
    restored_model, restored_processor, restored_labels_ko = reload_classifier(
        Path(OUTPUT_DIR) / "adapter"
    )

def analyze_expression(image):
    classifier = make_image_pipeline(restored_model, restored_processor)
    scores = classifier(expression_image(image), top_k=len(restored_labels_ko))
    return pd.DataFrame([{
        "표정": restored_labels_ko[item["label"]],
        "점수": item["score"],
    } for item in scores])


if UPLOAD_FROM_COLAB:
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("이미지 파일을 한 개 선택하세요.")
    with Image.open(BytesIO(next(iter(uploaded.values())))) as image:
        demo_image = ImageOps.exif_transpose(image).convert("RGB")
elif IMAGE_PATH:
    with Image.open(IMAGE_PATH) as image:
        demo_image = ImageOps.exif_transpose(image).convert("RGB")
elif "raw" in globals():
    demo_image = raw["test"][selected_ids[0]]["image"]
else:
    raise ValueError("IMAGE_PATH를 지정하거나 UPLOAD_FROM_COLAB을 켜세요.")

preview = demo_image.copy()
preview.thumbnail((360, 360))
display(preview)
expression_scores = analyze_expression(demo_image)
print("예측 표정:", expression_scores.iloc[0]["표정"])
display(expression_scores.style.format({"점수": "{:.1%}"}))

예측 표정: 기쁨


,표정,점수
0,기쁨,85.3%
1,두려움,4.4%
2,중립,3.1%
3,분노,2.6%
4,슬픔,2.5%
5,놀람,2.0%
6,혐오,0.2%
